In [ ]:
!pip install numpy pandas scikit-learn tensorflow

In [ ]:
import os
import sqlite3
import logging
import numpy as np
import pandas as pd

from sklearn.preprocessing import MinMaxScaler
from tensorflow.keras.models import Sequential, load_model
from tensorflow.keras.layers import LSTM, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping
from concurrent.futures import ThreadPoolExecutor, as_completed

In [ ]:
BASE_DIR = os.path.dirname(os.path.abspath(__file__))

DB_PATH = os.path.join(BASE_DIR, "data", "crypto_daily.db")
MODEL_DIR = os.path.join(BASE_DIR, "models")

LOOKBACK = 30
MAX_WORKERS = 3

os.makedirs(MODEL_DIR, exist_ok=True)

In [ ]:
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(message)s"
)
logger = logging.getLogger(__name__)

In [ ]:
def model_path(symbol):
    return os.path.join(MODEL_DIR, f"{symbol}.h5")

In [ ]:
def scaler_path(symbol):
    return os.path.join(MODEL_DIR, f"{symbol}_scaler.npy")

In [ ]:
def ensure_predictions_table():
    conn = sqlite3.connect(DB_PATH)
    conn.execute("""
        CREATE TABLE IF NOT EXISTS predictions (
            symbol TEXT,
            date TEXT,
            predicted_close REAL,
            PRIMARY KEY (symbol, date)
        );
    """)
    conn.commit()
    conn.close()

In [ ]:
def build_lstm_model(lookback, features):
    model = Sequential([
        LSTM(64, return_sequences=True, input_shape=(lookback, features)),
        Dropout(0.2),
        LSTM(32),
        Dense(1)
    ])
    model.compile(optimizer="adam", loss="mse")
    return model

In [ ]:
def get_last_prediction_date(conn, symbol):
    row = conn.execute(
        "SELECT MAX(date) FROM predictions WHERE symbol=?",
        (symbol,)
    ).fetchone()
    return pd.to_datetime(row[0]) if row and row[0] else None

In [ ]:
def process_coin(symbol, dry_run=False):
    try:
        conn = sqlite3.connect(DB_PATH, timeout=30)

        df = pd.read_sql("""
            SELECT date, open, high, low, close, volume
            FROM daily
            WHERE symbol=?
            ORDER BY date ASC
        """, conn, params=(symbol,))

        if len(df) < LOOKBACK + 1:
            logger.warning(f"{symbol}: not enough data")
            conn.close()
            return

        df["date"] = pd.to_datetime(df["date"])

        last_pred_date = get_last_prediction_date(conn, symbol)
        if last_pred_date is not None:
            df_new = df[df["date"] > last_pred_date]
        else:
            df_new = df.copy()

        if len(df_new) < LOOKBACK + 1:
            logger.info(f"{symbol}: no new data")
            conn.close()
            return

        #scale data
        cols = ["open", "high", "low", "close", "volume"]

        if os.path.exists(scaler_path(symbol)):
            scaler = np.load(scaler_path(symbol), allow_pickle=True).item()
            scaled = scaler.transform(df[cols])
        else:
            scaler = MinMaxScaler()
            scaled = scaler.fit_transform(df[cols])
            np.save(scaler_path(symbol), scaler)

        # relevant data only
        X, y, dates = [], [], []

        start_idx = len(df) - len(df_new) - LOOKBACK
        start_idx = max(start_idx, LOOKBACK)

        for i in range(start_idx, len(scaled)):
            X.append(scaled[i - LOOKBACK:i])
            y.append(scaled[i, 3])
            dates.append(df.iloc[i]["date"])

        X = np.array(X)
        y = np.array(y)

        if dry_run:
            logger.info(f"[DRY RUN] {symbol}: {len(X)} samples")
            conn.close()
            return

        #training a new model or loading it if it exists
        if os.path.exists(model_path(symbol)):
            model = load_model(model_path(symbol))
            epochs = 5
        else:
            model = build_lstm_model(LOOKBACK, X.shape[2])
            epochs = 30

        model.fit(
            X, y,
            epochs=epochs,
            batch_size=32,
            verbose=0,
            callbacks=[EarlyStopping(patience=3, restore_best_weights=True)]
        )

        model.save(model_path(symbol))

        #saved predictions
        preds = model.predict(X, verbose=0)

        temp = np.zeros((len(preds), 5))
        temp[:, 3] = preds[:, 0]
        inv_preds = scaler.inverse_transform(temp)[:, 3]

        rows = [
            (symbol, d.strftime("%Y-%m-%d"), float(p))
            for d, p in zip(dates, inv_preds)
        ]

        conn.executemany("""
            INSERT OR REPLACE INTO predictions (symbol, date, predicted_close)
            VALUES (?, ?, ?)
        """, rows)

        conn.commit()
        conn.close()

        logger.info(f"{symbol}: updated ({len(rows)} predictions)")

    except Exception as e:
        logger.error(f"{symbol}: failed → {e}")

In [ ]:
#parallel running
def run_pipeline(symbols, dry_run=False):
    with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
        futures = [
            executor.submit(process_coin, s, dry_run)
            for s in symbols
        ]
        for _ in as_completed(futures):
            pass

In [ ]:
if __name__ == "__main__":
    ensure_predictions_table()

    #test with a few coins
    TEST_COINS = ["BTC", "ETH"]
    run_pipeline(TEST_COINS, dry_run=True)


    run_pipeline(TEST_COINS, dry_run=False)


    # conn = sqlite3.connect(DB_PATH)
    # ALL_COINS = pd.read_sql(
    #     "SELECT DISTINCT symbol FROM daily",
    #     conn
    # )["symbol"].tolist()
    # conn.close()
    # run_pipeline(ALL_COINS)